In [1]:
import pandas as pd
import re
import os

In [2]:
def robust_slicer(text):
    if not isinstance(text, str) or len(text) < 10:
        return "N/A"

    # 1. Hallucination Cleanup: Kill markdown code blocks and generic impact tags
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
    text = re.sub(r'\[Impact\]:.*', '', text, flags=re.IGNORECASE | re.DOTALL)
    
    # 2. Header Extraction: Isolate exactly what is between ANALYSIS and IMPACT
    # Added 'Analysis' (case-insensitive) match
    analysis_match = re.search(r'ANALYSIS:(.*?)(?=IMPACT:|STOP:|VULNERABILITY:|$)', text, re.IGNORECASE | re.DOTALL)
    
    if analysis_match and len(analysis_match.group(1).strip()) > 5:
        core_content = analysis_match.group(1).strip()
    else:
        # Fallback: If no ANALYSIS header, take the whole text
        core_content = text

    # 3. Sentence Cleaning: Take the first 3 sentences 
    # specifically targets the "Truth" usually found at the start
    sentences = re.split(r'(?<=[.!?]) +', core_content)
    cleaned_text = " ".join(sentences[:3]).strip()

    # 4. Final Word Limit: Hard cut at 70 words
    words = cleaned_text.split()
    if len(words) > 70:
        cleaned_text = " ".join(words[:70]) + "..."

    return cleaned_text

In [3]:
# Execution Block
DRIVE_PATH = '/content/drive/MyDrive/Project/results'
for persona in ['naive', 'formal', 'expert']:
    input_file = f'{DRIVE_PATH}/rq2_{persona}_results.csv'
    output_file = f'{DRIVE_PATH}/rq2_{persona}_cleaned.csv'
    
    if os.path.exists(input_file):
        print(f"Slicing {persona.upper()} data...")
        df = pd.read_csv(input_file)
        
        # Apply the slicer to create the evaluation-ready column
        df['cleaned_explanation'] = df['generated_explanation'].apply(robust_slicer)
        
        df.to_csv(output_file, index=False)
        print(f"--- Saved Cleaned File: {output_file} ---")
    else:
        print(f"Warning: {input_file} not found. Skipping.")

Slicing NAIVE data...
--- Saved Cleaned File: /content/drive/MyDrive/Project/results/rq2_naive_cleaned.csv ---
Slicing FORMAL data...
--- Saved Cleaned File: /content/drive/MyDrive/Project/results/rq2_formal_cleaned.csv ---
Slicing EXPERT data...
--- Saved Cleaned File: /content/drive/MyDrive/Project/results/rq2_expert_cleaned.csv ---
